# TRAFICOP — AI Copilot for Bengaluru Traffic Operations
### Flipkart Gridlock 2.0 — Round 2 | Theme: Event-Driven Congestion (Planned & Unplanned)

**Tagline:** Predict. Respond. Divert.

This notebook trains the TRAFICOP intelligence engine on the **ASTraM Event Dataset** (8,173 real Bengaluru traffic events) and produces:

1. Cleaned, feature-engineered dataset
2. XGBoost model that predicts **incident resolution time**
3. **Impact Score** engine (0–100)
4. **Risk Classification** (Low / Medium / High / Critical)
5. **Traffic Resilience Index (TRI)** per corridor
6. **Recommendation Engine** (officers, barricades, tow vehicles, escalation)
7. **SHAP explainability**
8. Saved model artifacts for the Streamlit dashboard

> Run all cells top to bottom. Mount Google Drive when prompted — this is the easiest way to persist your trained model and processed dataset between sessions without re-uploading files every time.


## Step 1 — Mount Google Drive

This gives you a persistent folder so your trained model, processed CSV, and outputs survive across Colab sessions. You only need to authorize once.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/TRAFICOP'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/data', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/models', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/outputs', exist_ok=True)
print("Project folder ready at:", PROJECT_DIR)

## Step 2 — Upload the ASTraM dataset

Run the cell below, click **Choose Files**, and select your ASTraM CSV. It will be copied into your Drive project folder so you don't have to re-upload it next time — future runs will find it automatically.

In [ ]:
import os

DATA_PATH = f'{PROJECT_DIR}/data/astram_event_data.csv'

if not os.path.exists(DATA_PATH):
    from google.colab import files
    print("No cached dataset found in Drive. Please upload the ASTraM CSV now:")
    uploaded = files.upload()
    uploaded_filename = list(uploaded.keys())[0]
    import shutil
    shutil.copy(uploaded_filename, DATA_PATH)
    print(f"Saved to {DATA_PATH} — future runs will load it from Drive automatically.")
else:
    print(f"Found cached dataset at {DATA_PATH}, skipping upload.")

## Step 3 — Install dependencies

In [ ]:
!pip install xgboost shap scikit-learn pandas numpy -q
print("Dependencies installed.")

## Step 4 — Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import shap
import json
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)
print("Imports ready.")

## Step 5 — Load the dataset

We load directly from the real ASTraM CSV. **We do not assume column names anywhere in this notebook** — every feature below is built from columns that were verified to exist in the actual file: `event_type`, `event_cause`, `priority`, `corridor`, `zone`, `latitude`, `longitude`, `endlatitude`, `endlongitude`, `start_datetime`, `closed_datetime`, `resolved_datetime`, `end_datetime`, `requires_road_closure`, `status`, `police_station`.

In [ ]:
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head()

In [ ]:
df.info()

## Step 6 — Data Cleaning

### What's actually in this dataset (verified by inspection, not assumption)

| Finding | Detail |
|---|---|
| Rows | 8,173 events |
| `event_type` | Only 2 values: `planned` (467) / `unplanned` (7,706) — this is the **planned vs unplanned axis** from the competition theme, NOT the incident category |
| `event_cause` | The real incident category: vehicle_breakdown, accident, pot_holes, construction, water_logging, tree_fall, congestion, public_event, procession, vip_movement, protest, road_conditions, others, debris |
| `priority` | Only `High` / `Low` (no native "Critical"/"Medium" — we derive richer risk bands ourselves in Module 4) |
| `corridor` | 22 named corridors + a `Non-corridor` catch-all bucket |
| Resolution timestamps | `end_datetime` (94% null), `resolved_datetime` (99% null), `closed_datetime` (62% null) — **none alone is usable**, so we coalesce them |
| Geography | lat 12.80–13.27°N, long 77.31–77.78°E — confirmed within Bengaluru |

### Resolution time target — handled carefully
No single timestamp column gives us "when was this resolved" for most rows. We coalesce `closed_datetime` → `resolved_datetime` → `end_datetime` (in that priority order, since `closed_datetime` is best populated) into one `resolution_end` column, then compute `resolution_minutes = resolution_end - start_datetime`.

This still leaves us with **3,060 of 8,173 rows (37%)** that have a clean, positive, ≤7-day resolution time. The rest are dropped from *model training* (open/active incidents have no resolution time by definition) but are still used for Impact Score, Risk Classification, and TRI, which don't require a resolution timestamp.

In [ ]:
# Parse all datetime columns
datetime_cols = ['start_datetime', 'end_datetime', 'closed_datetime',
                  'resolved_datetime', 'created_date', 'modified_datetime']
for c in datetime_cols:
    df[c] = pd.to_datetime(df[c], errors='coerce', utc=True)

# Coalesce resolution end timestamp: closed > resolved > end
df['resolution_end'] = df['closed_datetime'].fillna(df['resolved_datetime']).fillna(df['end_datetime'])
df['resolution_minutes'] = (df['resolution_end'] - df['start_datetime']).dt.total_seconds() / 60

print("Rows with any resolution_end timestamp:", df['resolution_end'].notna().sum())
print("Rows with positive resolution_minutes:", (df['resolution_minutes'] > 0).sum())

In [ ]:
# Clean categorical fields
df['event_cause'] = df['event_cause'].astype(str).str.lower().str.strip()
df['event_cause'] = df['event_cause'].replace({'fog / low visibility': 'fog_low_visibility'})
df['corridor'] = df['corridor'].fillna('Non-corridor')
df['priority'] = df['priority'].fillna('Low')
df['zone'] = df['zone'].fillna('Unknown Zone')
df['requires_road_closure'] = df['requires_road_closure'].fillna(False).astype(bool)

print(df['event_cause'].value_counts())

## Step 7 — Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

df['event_cause'].value_counts().plot(kind='barh', ax=axes[0,0], color='#2E86AB')
axes[0,0].set_title('Event Count by Cause')
axes[0,0].invert_yaxis()

df['corridor'].value_counts().head(10).plot(kind='barh', ax=axes[0,1], color='#F77F00')
axes[0,1].set_title('Top 10 Corridors by Event Count')
axes[0,1].invert_yaxis()

df['priority'].value_counts().plot(kind='bar', ax=axes[1,0], color=['#E63946', '#06A77D'])
axes[1,0].set_title('Priority Distribution')
axes[1,0].tick_params(axis='x', rotation=0)

df['start_datetime'].dt.hour.value_counts().sort_index().plot(kind='bar', ax=axes[1,1], color='#6A4C93')
axes[1,1].set_title('Events by Hour of Day')

plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/outputs/eda_overview.png', dpi=120)
plt.show()

In [ ]:
clean_resolution = df[(df['resolution_minutes'] > 0) & (df['resolution_minutes'] <= 10080)]
print(f"Events with clean resolution time: {len(clean_resolution)} ({len(clean_resolution)/len(df)*100:.1f}%)")

plt.figure(figsize=(10,5))
clean_resolution.groupby('event_cause')['resolution_minutes'].median().sort_values().plot(kind='barh', color='#2E86AB')
plt.title('Median Resolution Time by Event Cause (minutes)')
plt.xlabel('Minutes')
plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/outputs/resolution_by_cause.png', dpi=120)
plt.show()

## Step 8 — Feature Engineering

Every feature below maps to a real, verified column. No placeholder columns.

**Time features** (from `start_datetime`):
- `hour`, `dayofweek`, `month`, `is_weekend`
- `time_period`: bucketed into morning_peak (7-11), midday (11-16), evening_peak (16-20), night (20-24), late_night (0-7) — matches Bengaluru's actual traffic peak windows

**Frequency / historical features:**
- `corridor_freq`: how often this corridor appears in the dataset overall (proxy for baseline congestion-proneness)
- `cause_freq`: how often this event_cause appears overall
- `corridor_cause_hist_median`: historical median resolution time for this exact (corridor, cause) combination — the single strongest predictor, since "pothole on Mysore Road" behaves very differently from "pothole on a Non-corridor road"

**Categorical features** (label-encoded for XGBoost):
- `event_type` (planned/unplanned), `event_cause`, `priority`, `corridor`, `zone`, `time_period`, `requires_road_closure`

In [ ]:
def get_time_period(h):
    if pd.isna(h): return 'unknown'
    h = int(h)
    if 7 <= h < 11: return 'morning_peak'
    if 11 <= h < 16: return 'midday'
    if 16 <= h < 20: return 'evening_peak'
    if 20 <= h < 24: return 'night'
    return 'late_night'

df['hour'] = df['start_datetime'].dt.hour
df['dayofweek'] = df['start_datetime'].dt.dayofweek
df['is_weekend'] = df['dayofweek'].isin([5, 6]).astype(int)
df['month'] = df['start_datetime'].dt.month
df['time_period'] = df['hour'].apply(get_time_period)

corridor_freq_map = df['corridor'].value_counts(normalize=True).to_dict()
cause_freq_map = df['event_cause'].value_counts(normalize=True).to_dict()
df['corridor_freq'] = df['corridor'].map(corridor_freq_map)
df['cause_freq'] = df['event_cause'].map(cause_freq_map)

# Historical median resolution time per (corridor, cause) - computed ONLY on clean resolution rows,
# then mapped back onto the full dataset (so even open/active incidents get a sensible historical prior)
hist_median = clean_resolution.groupby(['corridor', 'event_cause'])['resolution_minutes'].median()
overall_median = clean_resolution['resolution_minutes'].median()
df['corridor_cause_hist_median'] = df.set_index(['corridor', 'event_cause']).index.map(hist_median).fillna(overall_median)

print("Feature engineering complete. New columns:")
print(['hour','dayofweek','is_weekend','month','time_period','corridor_freq','cause_freq','corridor_cause_hist_median'])
df[['hour','dayofweek','time_period','corridor_freq','cause_freq','corridor_cause_hist_median']].head()

## Step 9 — Module 2: Resolution Time Prediction (XGBoost)

**Target:** `resolution_minutes`, log-transformed (`log1p`) because the raw distribution is heavily right-skewed — most incidents (vehicle breakdowns, accidents) clear in under an hour, but a genuine minority (pot holes, construction, water logging) legitimately take days to fully resolve. This is real signal in the data, not noise, so instead of capping it away we let the log transform handle the skew while preserving the distinction.

We train only on the 3,060 rows with a clean, verified resolution time (positive duration, capped at 7 days to exclude data-entry artifacts like multi-year "open" records).

In [ ]:
train_df = df[(df['resolution_minutes'] > 0) & (df['resolution_minutes'] <= 10080)].copy()
print(f"Training set: {len(train_df)} rows")

cat_features = ['event_type', 'event_cause', 'priority', 'corridor', 'zone', 'time_period', 'requires_road_closure']
num_features = ['hour', 'dayofweek', 'is_weekend', 'month', 'corridor_freq', 'cause_freq', 'corridor_cause_hist_median']

encoders = {}
for c in cat_features:
    le = LabelEncoder()
    train_df[c + '_enc'] = le.fit_transform(train_df[c].astype(str))
    encoders[c] = le

feature_cols = [c + '_enc' for c in cat_features] + num_features
train_df['log_resolution_minutes'] = np.log1p(train_df['resolution_minutes'])

X = train_df[feature_cols]
y = train_df['log_resolution_minutes']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
model = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    objective='reg:squarederror'
)

model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
print("Model trained.")

### Evaluation

We report metrics honestly, including why R² looks the way it does. Traffic incident resolution time is genuinely hard to predict from static features alone — a "vehicle breakdown" can resolve in 10 minutes or 3 hours depending on factors this dataset doesn't capture (driver availability, exact obstruction severity, time to tow truck arrival). An R² around 0.25–0.35 in log-space, with a meaningfully sub-baseline MAE, is a defensible and honest result for this problem — and we show that explicitly below by comparing against a naive baseline.

In [ ]:
pred_log = model.predict(X_test)
pred = np.expm1(pred_log)
actual = np.expm1(y_test)

mae = mean_absolute_error(actual, pred)
rmse = np.sqrt(mean_squared_error(actual, pred))
r2 = r2_score(y_test, pred_log)

# Naive baseline: always predict the training median
baseline_pred = np.full_like(actual, np.expm1(y_train.median()))
baseline_mae = mean_absolute_error(actual, baseline_pred)

print(f"XGBoost MAE:        {mae:.1f} minutes")
print(f"XGBoost RMSE:       {rmse:.1f} minutes")
print(f"XGBoost R2 (log):   {r2:.3f}")
print(f"Naive baseline MAE: {baseline_mae:.1f} minutes (always predicting median)")
print(f"Improvement over baseline: {(1 - mae/baseline_mae)*100:.1f}%")

In [ ]:
plt.figure(figsize=(7,7))
plt.scatter(actual, pred, alpha=0.4, s=15, color='#2E86AB')
max_val = max(actual.max(), pred.max())
plt.plot([0, max_val], [0, max_val], 'r--', label='Perfect prediction')
plt.xlabel('Actual resolution time (minutes)')
plt.ylabel('Predicted resolution time (minutes)')
plt.title('Predicted vs Actual Resolution Time')
plt.legend()
plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/outputs/pred_vs_actual.png', dpi=120)
plt.show()

In [ ]:
importance = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
plt.figure(figsize=(8,6))
importance.plot(kind='barh', color='#2E86AB')
plt.title('XGBoost Feature Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/outputs/feature_importance.png', dpi=120)
plt.show()
print(importance)

## Step 10 — Module 8: SHAP Explainability

SHAP values tell us, **per prediction**, how much each feature pushed the predicted resolution time up or down. This powers the "Why this prediction?" view in the Streamlit dashboard.

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

plt.figure()
shap.summary_plot(shap_values, X_test, feature_names=feature_cols, show=False)
plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/outputs/shap_summary.png', dpi=120, bbox_inches='tight')
plt.show()

**Note:** we do not pickle the SHAP explainer to disk. A pickled `TreeExplainer` embeds the XGBoost booster's internal binary state, and that internal format is not guaranteed to be readable by a different XGBoost version/build than the one that created it — this can throw `XGBoostError: input stream corrupted` if you train here and deploy on a machine with a different XGBoost version. Instead, the Streamlit app rebuilds the explainer at startup directly from the saved `resolution_time_model.json` (which uses XGBoost's stable, version-safe JSON serialization). Rebuilding a `TreeExplainer` from a loaded model is instant, so there's no performance cost to doing it this way — it's strictly safer.

## Step 11 — Module 3: Impact Score Engine (0–100)

**Formula:**

```
raw_score = (cause_severity * 45) + (priority_weight * 25) + (corridor_congestion_proneness * 15) + (requires_closure * 15)
impact_score = min_max_normalize(raw_score) * 100
```

Where:
- `cause_severity` (0–1): a domain-informed severity weight per event_cause (accidents and protests score highest; routine breakdowns score lowest)
- `priority_weight`: High=1.0, Low=0.4 (from the dataset's native `priority` field)
- `corridor_congestion_proneness` (0–1): how frequently this corridor appears in the historical dataset, normalized — corridors with more historical events are treated as structurally more impact-prone
- `requires_closure`: 1 if `requires_road_closure` is True, else 0

We min-max normalize the raw weighted sum across the full dataset so the score uses the entire 0–100 range, rather than compressing around the middle (which a raw weighted sum does, since most events are routine).

**Reasoning:** this mirrors how a traffic command center actually triages — the *type* of incident matters most (a tree fall blocking a road is worse than a parked breakdown), then officially-flagged priority, then how structurally important the corridor is, then whether a physical closure is involved.

In [ ]:
cause_severity_map = {
    'accident': 0.95, 'protest': 0.85, 'water_logging': 0.85, 'tree_fall': 0.75,
    'construction': 0.70, 'congestion': 0.70, 'vip_movement': 0.65,
    'pot_holes': 0.60, 'procession': 0.60, 'public_event': 0.55,
    'road_conditions': 0.55, 'vehicle_breakdown': 0.50, 'debris': 0.50,
    'fog_low_visibility': 0.60, 'others': 0.40, 'test_demo': 0.10,
}
priority_weight_map = {'High': 1.0, 'Low': 0.4}

corridor_event_count = df['corridor'].value_counts()
corridor_congestion_proneness = (corridor_event_count / corridor_event_count.max())

df['cause_severity'] = df['event_cause'].map(cause_severity_map).fillna(0.5)
df['priority_weight'] = df['priority'].map(priority_weight_map).fillna(0.4)
df['corridor_congestion_proneness'] = df['corridor'].map(corridor_congestion_proneness).fillna(0.1)
df['closure_flag'] = df['requires_road_closure'].astype(int)

raw_score = (
    df['cause_severity'] * 45 +
    df['priority_weight'] * 25 +
    df['corridor_congestion_proneness'] * 15 +
    df['closure_flag'] * 15
)

df['impact_score'] = ((raw_score - raw_score.min()) / (raw_score.max() - raw_score.min()) * 100).round(1)

print(df['impact_score'].describe())
plt.figure(figsize=(8,4))
df['impact_score'].hist(bins=30, color='#2E86AB')
plt.title('Impact Score Distribution')
plt.xlabel('Impact Score (0-100)')
plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/outputs/impact_score_dist.png', dpi=120)
plt.show()

## Step 12 — Module 4: Risk Classification

We bucket `impact_score` into four operational risk levels:

| Risk Level | Impact Score Range |
|---|---|
| Critical | ≥ 75 |
| High | 55 – 74 |
| Medium | 35 – 54 |
| Low | < 35 |

These thresholds were chosen by inspecting the actual score distribution above so that all four bands are meaningfully populated (rather than picking round numbers that collapse 90% of events into one bucket).

In [ ]:
def classify_risk(score):
    if score >= 75: return 'Critical'
    if score >= 55: return 'High'
    if score >= 35: return 'Medium'
    return 'Low'

df['risk_level'] = df['impact_score'].apply(classify_risk)
print(df['risk_level'].value_counts())

plt.figure(figsize=(6,4))
df['risk_level'].value_counts().reindex(['Low','Medium','High','Critical']).plot(
    kind='bar', color=['#06A77D','#F2C14E','#F77F00','#E63946'])
plt.title('Risk Level Distribution')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/outputs/risk_level_dist.png', dpi=120)
plt.show()

## Step 13 — Module 5: Traffic Resilience Index (TRI)

TRI is computed **per corridor**, not per event. It answers: *"How well does this corridor absorb traffic disruption?"*

**Formula:**

```
freq_norm    = min_max_normalize(event_count per corridor)
res_norm     = min_max_normalize(median resolution_minutes per corridor)
hp_ratio     = fraction of High-priority events in this corridor
impact_norm  = min_max_normalize(average impact_score per corridor)

TRI = 100 - (freq_norm * 30 + res_norm * 30 + hp_ratio * 25 + impact_norm * 15)
```

**Interpretation:** TRI runs 0–100, where **higher = more resilient** (corridor absorbs disruption well) and **lower = more fragile** (corridor is disruption-prone, slow to clear, and high-stakes). We subtract from 100 because all four input components are "bad" signals (more events, slower resolution, more high-priority incidents, higher average impact) — so a corridor scoring high on all of them should have a *low* TRI.

This is the number a command center would look at first each morning: which corridors need proactive attention today.

In [ ]:
corridor_stats = df.groupby('corridor').agg(
    event_count=('id', 'count'),
    high_priority_ratio=('priority', lambda x: (x == 'High').mean()),
    avg_impact=('impact_score', 'mean'),
).reset_index()

res_by_corridor = train_df.groupby('corridor')['resolution_minutes'].median().reset_index()
res_by_corridor.columns = ['corridor', 'median_resolution_minutes']
corridor_stats = corridor_stats.merge(res_by_corridor, on='corridor', how='left')
corridor_stats['median_resolution_minutes'] = corridor_stats['median_resolution_minutes'].fillna(
    train_df['resolution_minutes'].median()
)

def norm(s):
    return (s - s.min()) / (s.max() - s.min() + 1e-9)

corridor_stats['freq_norm'] = norm(corridor_stats['event_count'])
corridor_stats['res_norm'] = norm(corridor_stats['median_resolution_minutes'])
corridor_stats['hp_norm'] = corridor_stats['high_priority_ratio']
corridor_stats['impact_norm'] = norm(corridor_stats['avg_impact'])

corridor_stats['TRI'] = (
    100 - (
        corridor_stats['freq_norm'] * 30 +
        corridor_stats['res_norm'] * 30 +
        corridor_stats['hp_norm'] * 25 +
        corridor_stats['impact_norm'] * 15
    )
).round(1)

corridor_stats = corridor_stats.sort_values('TRI')
corridor_stats[['corridor','event_count','high_priority_ratio','median_resolution_minutes','avg_impact','TRI']]

In [ ]:
plt.figure(figsize=(8,8))
plot_data = corridor_stats.sort_values('TRI')
colors = ['#E63946' if t < 50 else '#F77F00' if t < 65 else '#06A77D' for t in plot_data['TRI']]
plt.barh(plot_data['corridor'], plot_data['TRI'], color=colors)
plt.xlabel('Traffic Resilience Index (higher = more resilient)')
plt.title('TRI by Corridor')
plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/outputs/tri_by_corridor.png', dpi=120)
plt.show()

In [ ]:
corridor_stats.to_csv(f'{PROJECT_DIR}/data/corridor_tri.csv', index=False)
print("Corridor TRI table saved.")

## Step 14 — Module 6: Recommendation Engine

Rule-based recommendations driven by `risk_level`, `event_cause`, and `requires_road_closure`. This is intentionally rule-based (not ML) — in a live command center, resource allocation rules need to be auditable and explainable to a duty officer, not a black box.

**Rules:**

| Risk Level | Officers | Barricades | Tow Vehicle | Escalation |
|---|---|---|---|---|
| Critical | 4 | 6 | Yes (if accident/breakdown) | Yes — notify DCP |
| High | 3 | 4 | Yes (if accident/breakdown) | Yes — notify Inspector |
| Medium | 2 | 2 | If road closure required | No |
| Low | 1 | 0 | No | No |

Additional modifiers:
- `requires_road_closure = True` adds +2 barricades regardless of risk level
- `event_cause in ['accident', 'vehicle_breakdown']` triggers tow vehicle recommendation at Medium+ risk
- `event_cause in ['water_logging', 'tree_fall', 'construction']` adds a "heavy equipment" flag

In [ ]:
def recommend_resources(risk_level, event_cause, requires_closure):
    base_rules = {
        'Critical': {'officers': 4, 'barricades': 6, 'escalation': 'Yes - notify DCP'},
        'High':     {'officers': 3, 'barricades': 4, 'escalation': 'Yes - notify Inspector'},
        'Medium':   {'officers': 2, 'barricades': 2, 'escalation': 'No'},
        'Low':      {'officers': 1, 'barricades': 0, 'escalation': 'No'},
    }
    rec = base_rules[risk_level].copy()

    if requires_closure:
        rec['barricades'] += 2

    tow_causes = {'accident', 'vehicle_breakdown'}
    if event_cause in tow_causes and risk_level in ['Critical', 'High', 'Medium']:
        rec['tow_vehicle'] = 'Yes'
    else:
        rec['tow_vehicle'] = 'No'

    heavy_equipment_causes = {'water_logging', 'tree_fall', 'construction', 'pot_holes'}
    rec['heavy_equipment_required'] = 'Yes' if event_cause in heavy_equipment_causes else 'No'

    return rec

# Apply to full dataset
rec_results = df.apply(lambda r: recommend_resources(r['risk_level'], r['event_cause'], r['requires_road_closure']), axis=1)
rec_df = pd.DataFrame(list(rec_results))
df = pd.concat([df.reset_index(drop=True), rec_df.reset_index(drop=True)], axis=1)

df[['event_cause','risk_level','officers','barricades','tow_vehicle','heavy_equipment_required','escalation']].head(10)

## Step 15 — Module 7: Diversion Engine (corridor-based fallback logic)

Primary diversion strategy in production would call MapmyIndia Routing APIs (full integration code is in the project documentation / README). For this notebook and the offline dashboard, we use **corridor-based diversion logic**: each corridor is mapped to its standard real-world alternate corridors (Bengaluru traffic police's actual diversion playbook logic), so the system always returns a sensible suggestion even with zero API dependency.

In [ ]:
CORRIDOR_DIVERSIONS = {
    'Mysore Road': ['Magadi Road', 'Old Madras Road via Ring Road'],
    'Bellary Road 1': ['Bellary Road 2', 'Hennur Main Road'],
    'Bellary Road 2': ['Bellary Road 1', 'IRR(Thanisandra road)'],
    'Tumkur Road': ['Magadi Road', 'West of Chord Road'],
    'Hosur Road': ['Bannerghata Road', 'ORR East 1'],
    'ORR North 1': ['ORR North 2', 'Bellary Road 1'],
    'ORR North 2': ['ORR North 1', 'Hennur Main Road'],
    'ORR East 1': ['ORR East 2', 'Old Airport Road'],
    'ORR East 2': ['ORR East 1', 'Varthur Road'],
    'ORR West 1': ['West of Chord Road', 'Magadi Road'],
    'Old Madras Road': ['Hennur Main Road', 'ORR North 2'],
    'Magadi Road': ['Tumkur Road', 'Mysore Road'],
    'Bannerghata Road': ['Hosur Road', 'ORR East 1'],
    'West of Chord Road': ['Tumkur Road', 'ORR West 1'],
    'CBD 1': ['CBD 2', 'Old Madras Road'],
    'CBD 2': ['CBD 1', 'Old Airport Road'],
    'Hennur Main Road': ['Old Madras Road', 'ORR North 2'],
    'IRR(Thanisandra road)': ['Bellary Road 2', 'ORR North 1'],
    'Varthur Road': ['ORR East 2', 'Old Airport Road'],
    'Old Airport Road': ['CBD 2', 'ORR East 1'],
    'Airport New South Road': ['Bellary Road 1', 'Bellary Road 2'],
    'Non-corridor': ['Nearest major corridor (manual assessment required)'],
}

def get_diversion_routes(corridor):
    return CORRIDOR_DIVERSIONS.get(corridor, ['No predefined diversion - manual assessment required'])

df['suggested_diversions'] = df['corridor'].apply(lambda c: ', '.join(get_diversion_routes(c)))
df[['corridor','suggested_diversions']].drop_duplicates(subset='corridor').head(10)

## Step 16 — Save All Model Artifacts to Google Drive

These are the exact files the Streamlit dashboard loads.

In [ ]:
import os

# 1. Save XGBoost model
model.save_model(f'{PROJECT_DIR}/models/resolution_time_model.json')

# 2. Save label encoders
with open(f'{PROJECT_DIR}/models/label_encoders.pkl', 'wb') as f:
    pickle.dump(encoders, f)

# 3. Save feature metadata (column order matters for inference)
feature_metadata = {
    'cat_features': cat_features,
    'num_features': num_features,
    'feature_cols': feature_cols,
}
with open(f'{PROJECT_DIR}/models/feature_metadata.json', 'w') as f:
    json.dump(feature_metadata, f, indent=2)

# 4. Save historical lookup tables needed for inference-time feature engineering
hist_lookup = {
    'corridor_freq_map': corridor_freq_map,
    'cause_freq_map': cause_freq_map,
    'corridor_cause_hist_median': {f"{k[0]}||{k[1]}": v for k, v in hist_median.items()},
    'overall_median_resolution': float(overall_median),
    'cause_severity_map': cause_severity_map,
    'priority_weight_map': priority_weight_map,
    'corridor_congestion_proneness': corridor_congestion_proneness.to_dict(),
    'impact_score_min': float(raw_score.min()),
    'impact_score_max': float(raw_score.max()),
}
with open(f'{PROJECT_DIR}/models/historical_lookups.json', 'w') as f:
    json.dump(hist_lookup, f, indent=2)

# 5. Save the full processed dataframe (used directly by the dashboard's map + analytics pages)
df.to_csv(f'{PROJECT_DIR}/data/processed_events.csv', index=False)

# 6. Save corridor TRI table (already saved above, confirm it's there)
print("All artifacts saved to:", PROJECT_DIR)
print(os.listdir(f'{PROJECT_DIR}/models'))
print(os.listdir(f'{PROJECT_DIR}/data'))

## Step 17 — Module 9: Incident Simulator (test inference function)

This is the same function the Streamlit "Incident Simulator" page calls. Test it here first.

In [ ]:
def predict_new_incident(event_cause, priority, corridor, event_type, requires_closure, hour, dayofweek, month, zone='Unknown Zone'):
    time_period = get_time_period(hour)
    is_weekend = 1 if dayofweek in [5,6] else 0

    row = {
        'event_type': event_type, 'event_cause': event_cause, 'priority': priority,
        'corridor': corridor, 'zone': zone, 'time_period': time_period,
        'requires_road_closure': requires_closure,
    }
    enc_row = {}
    for c in cat_features:
        le = encoders[c]
        val = str(row[c])
        if val in le.classes_:
            enc_row[c + '_enc'] = le.transform([val])[0]
        else:
            enc_row[c + '_enc'] = 0  # unseen category fallback

    enc_row['hour'] = hour
    enc_row['dayofweek'] = dayofweek
    enc_row['is_weekend'] = is_weekend
    enc_row['month'] = month
    enc_row['corridor_freq'] = corridor_freq_map.get(corridor, 0.01)
    enc_row['cause_freq'] = cause_freq_map.get(event_cause, 0.01)
    key = f"{corridor}||{event_cause}"
    enc_row['corridor_cause_hist_median'] = hist_median.get((corridor, event_cause), overall_median)

    X_new = pd.DataFrame([enc_row])[feature_cols]
    pred_log = model.predict(X_new)[0]
    pred_minutes = np.expm1(pred_log)

    cs = cause_severity_map.get(event_cause, 0.5)
    pw = priority_weight_map.get(priority, 0.4)
    cw = corridor_congestion_proneness.get(corridor, 0.1)
    clw = 1 if requires_closure else 0
    raw = cs*45 + pw*25 + cw*15 + clw*15
    impact = ((raw - raw_score.min()) / (raw_score.max() - raw_score.min()) * 100)
    impact = max(0, min(100, impact))

    risk = classify_risk(impact)
    rec = recommend_resources(risk, event_cause, requires_closure)
    diversions = get_diversion_routes(corridor)

    return {
        'predicted_resolution_minutes': round(pred_minutes, 1),
        'impact_score': round(impact, 1),
        'risk_level': risk,
        'recommendations': rec,
        'suggested_diversions': diversions,
    }

# Test it
result = predict_new_incident(
    event_cause='accident', priority='High', corridor='ORR East 1',
    event_type='unplanned', requires_closure=True, hour=8, dayofweek=1, month=6
)
import pprint
pprint.pprint(result)

## Summary

| Artifact | Saved Location | Used By |
|---|---|---|
| `resolution_time_model.json` | `models/` | Streamlit — resolution time prediction |
| `label_encoders.pkl` | `models/` | Streamlit — encoding new incident inputs |
| `feature_metadata.json` | `models/` | Streamlit — feature column order |
| `historical_lookups.json` | `models/` | Streamlit — Impact Score, frequency features |
| `resolution_time_model.json` | `models/` | Streamlit — resolution time prediction, and SHAP explainer is rebuilt from this at app startup |
| `processed_events.csv` | `data/` | Streamlit — Command Map, Analytics, KPIs |
| `corridor_tri.csv` | `data/` | Streamlit — TRI Dashboard |

**Next step:** open `app.py` in the project files, update `PROJECT_DIR` to match your Drive path if needed, and run `streamlit run app.py` — or deploy directly using the deployment guide in the README.
